In [5]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [6]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по верблюдам v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Верблюды
256,АЛМАТИНСКАЯ ОБЛАСТЬ,2024-03-01,12.50
1165,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2018-08-01,215.90
809,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2021-12-01,25.50
925,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2021-06-01,0.50
987,МАНГИСТАУСКАЯ ОБЛАСТЬ,2016-01-01,166.92
678,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2020-11-01,1.40
248,АЛМАТИНСКАЯ ОБЛАСТЬ,2023-07-01,39.65
583,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2023-06-01,16.02
28,АКТЮБИНСКАЯ ОБЛАСТЬ,2015-04-01,115.59
199,АЛМАТИНСКАЯ ОБЛАСТЬ,2019-04-01,26.81


In [7]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Верблюды - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКТЮБИНСКАЯ ОБЛАСТЬ,2.23,4.04,2.34,3.77,11.96,19.19,HW,MAPE,2.23,3.77
1,АТЫРАУСКАЯ ОБЛАСТЬ,13.84,75.17,16.87,88.97,28.43,171.11,HW,MAPE,13.84,75.17
2,МАНГИСТАУСКАЯ ОБЛАСТЬ,32.34,115.97,43.24,129.42,33.17,116.67,HW,MAPE,32.34,115.97
3,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,NaN,239.23,72.67,152.43,46.91,133.71,Prophet,MAPE,46.91,133.71
4,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,20.88,58.47,23.49,62.72,17.35,63.03,Prophet,MAPE,17.35,58.47
5,АЛМАТИНСКАЯ ОБЛАСТЬ,NaN,11.06,59.56,10.36,95.35,13.75,SARIMA,MAPE,59.56,10.36
6,ЖАМБЫЛСКАЯ ОБЛАСТЬ,7.23,2.00,7.06,1.89,39.58,13.13,SARIMA,MAPE,7.06,1.89
7,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,NaN,1.00,7.25,1.01,33.83,2.89,SARIMA,MAPE,7.25,1.00
8,КАРАГАНДИНСКАЯ ОБЛАСТЬ,NaN,2.89,49.40,3.15,73.11,3.77,SARIMA,MAPE,49.40,2.89


In [10]:
actual_aug = pd.read_excel("Верблюды 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Верблюды"] = (actual_aug["Верблюды"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Верблюды обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Верблюды
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,NaN
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,5.90
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,34.80
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,166.10
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,6.50
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,11.50
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,NaN
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,NaN
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,61.30
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,157.90


In [11]:
# === настройки ===
TARGET = "Верблюды"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [12]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [13]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [14]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [15]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [16]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [ ]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Верблюды - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

18:23:50 - cmdstanpy - INFO - Chain [1] start processing
18:23:50 - cmdstanpy - INFO - Chain [1] done processing
18:23:50 - cmdstanpy - INFO - Chain [1] start processing
18:23:50 - cmdstanpy - INFO - Chain [1] done processing
